In [5]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ytdemtalk_microdynamics_reimplementation

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ytdemtalk_microdynamics_reimplementation


In [6]:
!pip install -r requirements.txt
!pip install --no-deps mediapipe==1.1.1 scenedetect==0.7.2
!pip install certifi

  Attempting uninstall: absl-py
    Found existing installation: absl-py 2.0.0
    Uninstalling absl-py-2.0.0:
      Successfully uninstalled absl-py-2.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sounddevice]


In [11]:
import re
p = "config.yaml"; s = open(p).read()
s = re.sub(r"(video_root:\s*)\S+", r"\g<1>.", s)
open(p, "w").write(s)
!grep -n "video_root\|local_video_cache" config.yaml

3:  video_root: .               # folder containing videos/Dementia/... and videos/Normal/... (see metadata.csv video_path)
4:  local_video_cache: /content/videos_local   # fast local disk; each video is copied here before processing


In [12]:
!PYTHONPATH=. python scripts/check_setup.py

python 3.13.18 (reference 3.13.18)
ok  numpy        2.4.6  (reference 2.4.6)
ok  pandas       2.2.3  (reference 2.2.3)
ok  scipy        1.16.3  (reference 1.16.3)
ok  sklearn      1.6.1  (reference 1.6.1)
ok  cv2          5.0.0  (reference 5.0.0)
I0000 00:00:1795627019.285416    3401 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795627019.365588    3401 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795627022.487336    3401 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
ok  mediapipe    1.1.1  (reference 1.1.1)
ok  scenedetect  0.7.2  (reference 0.7.2)
ok  tabulate     0.9.0  (reference 0.9.0)
ok  ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers  (re

In [13]:
!mkdir -p cache/frames cache/raw cache/cuts features results logs
!PYTHONPATH=. python scripts/detect_borders.py

1 {'video_id': 'uAlkCMfTASQ', 'W': 640, 'H': 360, 'crop_w': 640, 'crop_h': 360, 'crop_x': 0, 'crop_y': 0, 'content_frac': 1.0, 'n_samples': 5}
2 {'video_id': 'iJJerSu8DxE', 'W': 1920, 'H': 1080, 'crop_w': 1602, 'crop_h': 1080, 'crop_x': 158, 'crop_y': 0, 'content_frac': 0.8344, 'n_samples': 5}
3 {'video_id': 'E09agP3swYs', 'W': 1920, 'H': 1080, 'crop_w': 1920, 'crop_h': 1080, 'crop_x': 0, 'crop_y': 0, 'content_frac': 1.0, 'n_samples': 5}
4 {'video_id': '195rRDIN6_Y', 'W': 1280, 'H': 720, 'crop_w': 1280, 'crop_h': 720, 'crop_x': 0, 'crop_y': 0, 'content_frac': 1.0, 'n_samples': 5}
5 {'video_id': 'o73Ls8iyVOQ', 'W': 640, 'H': 480, 'crop_w': 640, 'crop_h': 360, 'crop_x': 0, 'crop_y': 60, 'content_frac': 0.75, 'n_samples': 5}
6 {'video_id': 'm0VOv_ZV-Xc', 'W': 640, 'H': 360, 'crop_w': 640, 'crop_h': 360, 'crop_x': 0, 'crop_y': 0, 'content_frac': 1.0, 'n_samples': 5}
7 {'video_id': '4qI8djrKniM', 'W': 1080, 'H': 1920, 'crop_w': 1080, 'crop_h': 1920, 'crop_x': 0, 'crop_y': 0, 'content_frac':

In [14]:
!nohup env PYTHONPATH=. python scripts/extract.py --all --shard 0 --nshards 2 > logs/extract_0.log 2>&1 &
!nohup env PYTHONPATH=. python scripts/extract.py --all --shard 1 --nshards 2 > logs/extract_1.log 2>&1 &

In [15]:
import time, subprocess
while True:
    out = subprocess.run("date +%H:%M; ls cache/frames/*.parquet | wc -l; "
                         "grep -h 'progress\\|STOP\\|Error' logs/extract_0.log logs/extract_1.log | tail -2; "
                         "ps aux | grep 'scripts/extract.py' | grep -v grep | wc -l",
                         shell=True, capture_output=True, text=True).stdout
    print(out, flush=True)
    if "STOP" in out or out.strip().endswith("\n0"):
        print(">>> Finished or stopped: check the output"); break
    time.sleep(600)

17:50
0
2

18:00
10
17:58:55 [w1] progress 4/99
18:00:41 [w1] progress 5/99
2

18:10
23
18:06:55 [w1] progress 10/99
18:09:32 [w1] progress 11/99
2

18:20
33
18:17:42 [w1] progress 15/99
18:19:43 [w1] progress 16/99
2

18:30
41
18:26:26 [w1] progress 19/99
18:29:02 [w1] progress 20/99
2

18:40
55
18:39:05 [w1] progress 26/99
18:40:39 [w1] progress 27/99
2

18:50
62
18:44:44 [w1] progress 29/99
18:47:36 [w1] progress 30/99
2

19:00
69
18:56:56 [w1] progress 33/99
18:59:46 [w1] progress 34/99
2

19:10
78
19:05:33 [w1] progress 37/99
19:09:36 [w1] progress 38/99
2

19:20
86
19:18:13 [w1] progress 41/99
19:20:47 [w1] progress 42/99
2

19:30
99
19:28:55 [w1] progress 48/99
19:30:19 [w1] progress 49/99
2

19:40
106
19:36:08 [w1] progress 51/99
19:38:43 [w1] progress 52/99
2

19:50
113
19:46:23 [w1] progress 55/99
19:49:05 [w1] progress 56/99
2

20:00
122
19:58:38 [w1] progress 60/99
20:00:20 [w1] progress 61/99
2

20:10
128
20:06:08 [w1] progress 63/99
20:09:29 [w1] progress 64/99
2

20:20
1

In [16]:
!PYTHONPATH=. python scripts/build_features.py

20/199 clips, 5 s
40/199 clips, 9 s
60/199 clips, 15 s
80/199 clips, 20 s
100/199 clips, 25 s
120/199 clips, 31 s
140/199 clips, 37 s
160/199 clips, 46 s
180/199 clips, 51 s
rule=splices_only: 5782 usable windows; 194 / 199 clips with >= 3 windows
excluded clips:
   video_id        label person_id orig_split  n_windows  valid_frac
L7ESquhxFeM     dementia       P06       test          0    0.978836
SOlXTTqW2kA     dementia       P16      train          1    0.401184
zZp4R12V184 Normotypical       P42      train          0    0.257778
q5gl8hmG5AY Normotypical       P45      train          0    0.212593
mmQcX6HpCGs Normotypical       P72      train          2    0.952593
                         clips  persons
orig_split label                       
test       Normotypical     17       17
           dementia          9        8
train      Normotypical     80       80
           dementia         88       14


In [17]:
!PYTHONPATH=. python scripts/protocol_a.py

RF {'AUROC': 0.817, 'AP': 0.717, 'F1': 0.621, 'Acc': 0.577, 'thr': 0.485, 'recall': 1.0, 'specificity': 0.353} T 20.0 PCA 10
GBDT {'AUROC': 0.712, 'AP': 0.589, 'F1': 0.571, 'Acc': 0.654, 'thr': 0.481, 'recall': 0.667, 'specificity': 0.647} T 20.0 PCA 10
ADA {'AUROC': 0.837, 'AP': 0.776, 'F1': 0.632, 'Acc': 0.731, 'thr': 0.499, 'recall': 0.667, 'specificity': 0.765} T 20.0 PCA 10
# Protocol A (orig_split train/test): Table 1 format, paper vs ours

Test split: 26 clips (9 dementia clips / 8 persons, 17 controls). Validation part of train: 58 clips (45 dementia / 5 persons, 13 controls). Fit part: 110 clips. Threshold = Youden J on the validation part, after temperature scaling.

| Method | | AUROC | AP | F1 | Acc | Thr |
|---|---|---|---|---|---|---|
| Stats+PCA+RF | paper | 0.953 | 0.961 | 0.851 | 0.857 | 0.636 |
| | **ours** | **0.817** | **0.717** | **0.621** | **0.577** | **0.485** |
| Stats+PCA+GBDT | paper | 0.893 | 0.925 | 0.844 | 0.846 | 0.522 |
| | **ours** | **0.712** | **0.589